In [1]:
from ultralytics import YOLO
import cv2
import os

# Load model
model = YOLO("best.pt")

input_dir = r"C:\Users\uzaai\OneDrive - Higher Education Commission\FYP_AI_Drone\input"
output_dir = "runs/detect/predict"
os.makedirs(output_dir, exist_ok=True)

CONF_THRES = 0.25
MIN_AREA_RATIO = 0.04  # ignore far objects

for img_name in os.listdir(input_dir):
    img_path = os.path.join(input_dir, img_name)
    print(f"\nProcessing: {img_path}")

    results = model(img_path, conf=CONF_THRES)

    for r in results:
        annotated = r.plot()
        h, w, _ = annotated.shape
        image_area = h * w

        LEFT_X = int(w * 0.33)
        RIGHT_X = int(w * 0.66)

        left_blocked = False
        center_blocked = False
        right_blocked = False

        if r.boxes:
            for box in r.boxes:
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                box_area = (x2 - x1) * (y2 - y1)

                # Ignore far objects
                if box_area / image_area < MIN_AREA_RATIO:
                    continue

                cx = (x1 + x2) // 2

                if cx < LEFT_X:
                    left_blocked = True
                elif cx > RIGHT_X:
                    right_blocked = True
                else:
                    center_blocked = True

        # FINAL DECISION
        if not center_blocked:
            action = "⬆️ MOVE FORWARD"
        elif not left_blocked:
            action = "⬅️ TURN LEFT"
        elif not right_blocked:
            action = "➡️ TURN RIGHT"
        else:
            action = "⛔ HOVER"

        # Overlay decision
        cv2.putText(
            annotated, action, (20, 40),
            cv2.FONT_HERSHEY_SIMPLEX, 1,
            (0, 255, 0), 3
        )

        save_path = os.path.join(output_dir, img_name)
        cv2.imwrite(save_path, annotated)

        print(f"Action: {action}")
        print(f"Saved: {save_path}")



Processing: C:\Users\uzaai\OneDrive - Higher Education Commission\FYP_AI_Drone\input\t1.jpg

image 1/1 C:\Users\uzaai\OneDrive - Higher Education Commission\FYP_AI_Drone\input\t1.jpg: 384x640 3 pedestrians, 2 cars, 2 trucks, 219.6ms
Speed: 13.8ms preprocess, 219.6ms inference, 18.2ms postprocess per image at shape (1, 3, 384, 640)
Action: ⬆️ MOVE FORWARD
Saved: runs/detect/predict\t1.jpg

Processing: C:\Users\uzaai\OneDrive - Higher Education Commission\FYP_AI_Drone\input\t2.jpg

image 1/1 C:\Users\uzaai\OneDrive - Higher Education Commission\FYP_AI_Drone\input\t2.jpg: 448x640 17 cars, 161.0ms
Speed: 7.6ms preprocess, 161.0ms inference, 4.2ms postprocess per image at shape (1, 3, 448, 640)
Action: ⬆️ MOVE FORWARD
Saved: runs/detect/predict\t2.jpg

Processing: C:\Users\uzaai\OneDrive - Higher Education Commission\FYP_AI_Drone\input\t3.jpg

image 1/1 C:\Users\uzaai\OneDrive - Higher Education Commission\FYP_AI_Drone\input\t3.jpg: 448x640 (no detections), 104.6ms
Speed: 4.7ms preprocess,

RealTime Navigation

In [1]:
from ultralytics import YOLO
import cv2

# Load model
model = YOLO("best.pt")

CONF_THRESH = 0.5
MIN_AREA_RATIO = 0.05

# Open webcam / video
cap = cv2.VideoCapture(0)  # 0 = webcam | replace with video path if needed

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    h, w, _ = frame.shape
    image_area = h * w

    CENTER_X1 = int(w * 0.4)
    CENTER_X2 = int(w * 0.6)

    results = model(frame, conf=CONF_THRESH, verbose=False)

    action = "⬆️ MOVE FORWARD"

    for r in results:
        if r.boxes:
            for box in r.boxes:
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                box_area = (x2 - x1) * (y2 - y1)

                if box_area / image_area < MIN_AREA_RATIO:
                    continue

                cx = (x1 + x2) // 2

                if CENTER_X1 <= cx <= CENTER_X2:
                    action = "⚠️ TURN LEFT"
                    break

        frame = r.plot()

    cv2.putText(frame, action, (20, 40),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

    cv2.imshow("Autonomous Drone Vision", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


antiflicker logic


In [ ]:
from ultralytics import YOLO
import cv2
import os

# =======================
# CONFIG
# =======================
MODEL_PATH = "best.pt"
INPUT_DIR = r"C:\Users\uzaai\OneDrive - Higher Education Commission\FYP_AI_Drone\input"
OUTPUT_DIR = "runs/detect/predict"

CONF_THRESH = 0.2
MIN_AREA_RATIO = 0.05        # ignore far/small objects
FRAME_CONFIRM = 3            # temporal stability

# =======================
# INIT
# =======================
os.makedirs(OUTPUT_DIR, exist_ok=True)
model = YOLO(MODEL_PATH)

danger_frames = 0

# =======================
# PROCESS IMAGES
# =======================
for img_name in os.listdir(INPUT_DIR):
    img_path = os.path.join(INPUT_DIR, img_name)
    print(f"\nProcessing: {img_path}")

    results = model(img_path, conf=CONF_THRESH)

    for r in results:
        annotated = r.plot()
        h, w, _ = annotated.shape
        image_area = h * w

        CENTER_X1 = int(w * 0.4)
        CENTER_X2 = int(w * 0.6)

        current_danger = False

        if r.boxes:
            for box in r.boxes:
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                box_area = (x2 - x1) * (y2 - y1)
                cx = (x1 + x2) // 2

                # Only consider BIG + CENTER obstacles
                if box_area / image_area > MIN_AREA_RATIO:
                    if CENTER_X1 < cx < CENTER_X2:
                        current_danger = True
                        break

        # =======================
        # TEMPORAL FILTER
        # =======================
        if current_danger:
            danger_frames += 1
        else:
            danger_frames = max(0, danger_frames - 1)

        # =======================
        # DECISION
        # =======================
        if danger_frames >= FRAME_CONFIRM:
            action = "⚠️ TURN LEFT"
        else:
            action = "⬆️ MOVE FORWARD"

        # =======================
        # OUTPUT
        # =======================
        save_path = os.path.join(OUTPUT_DIR, img_name)
        cv2.imwrite(save_path, annotated)

        print(f"Action: {action}")
        print(f"Saved: {save_path}")



Processing: C:\Users\uzaai\OneDrive - Higher Education Commission\FYP_AI_Drone\input\t1.jpg

image 1/1 C:\Users\uzaai\OneDrive - Higher Education Commission\FYP_AI_Drone\input\t1.jpg: 384x640 3 pedestrians, 4 cars, 3 trucks, 156.1ms
Speed: 48.4ms preprocess, 156.1ms inference, 2.3ms postprocess per image at shape (1, 3, 384, 640)
Action: ⬆️ MOVE FORWARD
Saved: runs/detect/predict\t1.jpg

Processing: C:\Users\uzaai\OneDrive - Higher Education Commission\FYP_AI_Drone\input\t2.jpg

image 1/1 C:\Users\uzaai\OneDrive - Higher Education Commission\FYP_AI_Drone\input\t2.jpg: 448x640 25 cars, 192.0ms
Speed: 7.2ms preprocess, 192.0ms inference, 1.5ms postprocess per image at shape (1, 3, 448, 640)
Action: ⬆️ MOVE FORWARD
Saved: runs/detect/predict\t2.jpg

Processing: C:\Users\uzaai\OneDrive - Higher Education Commission\FYP_AI_Drone\input\t3.jpg

image 1/1 C:\Users\uzaai\OneDrive - Higher Education Commission\FYP_AI_Drone\input\t3.jpg: 448x640 (no detections), 117.3ms
Speed: 4.8ms preprocess, 

low battery logic

In [ ]:
from ultralytics import YOLO
import cv2
import os
import time

# =======================
# CONFIG
# =======================
MODEL_PATH = "best.pt"
INPUT_DIR = r"C:\Users\uzaai\OneDrive - Higher Education Commission\FYP_AI_Drone\input"
OUTPUT_DIR = "runs/detect/predict"

CONF_THRESH = 0.2
MIN_AREA_RATIO = 0.05
FRAME_CONFIRM = 3

BATTERY_START = 100
BATTERY_DRAIN = 2          # % per frame
LOW_BATTERY = 20

# =======================
# INIT
# =======================
os.makedirs(OUTPUT_DIR, exist_ok=True)
model = YOLO(MODEL_PATH)

danger_frames = 0
battery = BATTERY_START
HOME_POSITION = True       # simulated GPS lock

# =======================
# PROCESS
# =======================
for img_name in os.listdir(INPUT_DIR):
    img_path = os.path.join(INPUT_DIR, img_name)
    print(f"\nProcessing: {img_path}")

    results = model(img_path, conf=CONF_THRESH)

    for r in results:
        annotated = r.plot()
        h, w, _ = annotated.shape
        image_area = h * w

        CENTER_X1 = int(w * 0.4)
        CENTER_X2 = int(w * 0.6)

        current_danger = False

        if r.boxes:
            for box in r.boxes:
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                cx = (x1 + x2) // 2
                area = (x2 - x1) * (y2 - y1)

                if area / image_area > MIN_AREA_RATIO:
                    if CENTER_X1 < cx < CENTER_X2:
                        current_danger = True
                        break

        # =======================
        # TEMPORAL FILTER
        # =======================
        if current_danger:
            danger_frames += 1
        else:
            danger_frames = max(0, danger_frames - 1)

        # =======================
        # BATTERY UPDATE
        # =======================
        battery -= BATTERY_DRAIN
        battery = max(battery, 0)

        # =======================
        # DECISION LOGIC
        # =======================
        if battery <= LOW_BATTERY:
            if HOME_POSITION:
                action = "🔁 RETURN TO HOME"
            else:
                action = "⬇️ EMERGENCY LAND"

        elif danger_frames >= FRAME_CONFIRM:
            action = "⚠️ AVOID OBSTACLE (TURN LEFT)"

        else:
            action = "⬆️ MOVE FORWARD"

        # =======================
        # DISPLAY INFO
        # =======================
        cv2.putText(
            annotated,
            f"Battery: {battery}%",
            (20, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (0, 255, 0) if battery > LOW_BATTERY else (0, 0, 255),
            2
        )

        cv2.putText(
            annotated,
            f"Action: {action}",
            (20, 80),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.9,
            (255, 255, 255),
            2
        )

        save_path = os.path.join(OUTPUT_DIR, img_name)
        cv2.imwrite(save_path, annotated)

        print(f"Battery: {battery}% | Action: {action}")
        print(f"Saved: {save_path}")

        time.sleep(0.2)   # simulate real-time
